In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
training_data = pd.read_csv("/kaggle/input/playground-series-s5e12/train.csv", index_col = 0)
training_data.head()

In [ ]:
training_data.tail(10)

In [ ]:
training_data.shape

In [ ]:
training_data.info()

In [ ]:
training_data.describe().T

### Checking the Number of Duplicates

In [ ]:
training_data.duplicated().sum()

In [ ]:
training_data.isnull().sum()

### Checking the Class imbalance in the dataset

In [ ]:
training_data['diagnosed_diabetes'].value_counts()

In [ ]:
training_data['diagnosed_diabetes'].value_counts(normalize=True)

### Univariate Distribution

In [ ]:
training_data.hist(figsize=(20, 15), bins=50, xlabelsize=8, ylabelsize=8)

In [ ]:
categorical_cols = ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history']
numeric_cols = [col for col, dtype in training_data.items() if col not in categorical_cols]

In [ ]:
plt.figure(figsize=(15,10))
for i, col in enumerate(numeric_cols,1):
    plt.subplot(5,4, i)
    sns.histplot(training_data[col], kde=True)
    plt.title(col)
plt.tight_layout()

In [ ]:
plot_count = 0
plt.figure(figsize=(20, 12))
for cat in categorical_cols:
    ax = plt.subplot(3, 3, plot_count+1)
    sns.countplot(data=training_data, x=cat)
    plot_count += 1
plt.show()

### Multivariate Distribution

In [ ]:
plt.figure(figsize=(18,15))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(5, 4, i)
    sns.histplot(data=training_data, x=col, hue='diagnosed_diabetes', kde=True, element="step")
    plt.title(f"{col} distribution by diagnosed_diabetes")
plt.tight_layout()

### Computing the Group stats

In [ ]:
training_data.groupby('diagnosed_diabetes')[numeric_cols].agg(['mean','median','std']).T

### Correlation matrix & Feature Relationship

In [ ]:
plt.figure(figsize=(10,8))
corr = training_data[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title("Correlation matrix")

In [ ]:
df = training_data.copy()

In [ ]:
numerical_features = df.select_dtypes(include=['int64','float64']).columns.tolist()
numerical_features = [c for c in numerical_features if c != 'diagnosed_diabetes']


categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

In [ ]:
X = df.drop(columns=['diagnosed_diabetes'])
y = df['diagnosed_diabetes']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

### Handling Outliers for Numerical Features

In [ ]:
def clip_outliers(series, k=1.5):
    q1, q3 = series.quantile([0.25,0.75])
    iqr = q3 - q1
    lower = q1 - k*iqr
    upper = q3 + k*iqr
    return series.clip(lower, upper)

for col in numerical_features:
    X[col] = clip_outliers(X[col])

#### Creating meaningful features

##### Creating Interactive features

In [ ]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # BMI + Age interaction
        X['bmi_age_interaction'] = X['bmi'] * X['age']

        # Blood pressure ratio (strongly correlated with metabolic syndrome)
        X['bp_ratio'] = X['systolic_bp'] / X['diastolic_bp']

        # Sedentary index
        X['sedentary_index'] = X['screen_time_hours_per_day'] / X['physical_activity_minutes_per_week'].replace(0,1)

        #Blood Pressure Interaction
        X['bp_interaction'] = X['systolic_bp'] * X['diastolic_bp']

        #Lipid Profile Ration
        X['chol_hdl_ratio'] = X['cholesterol_total'] / X['hdl_cholesterol']

        # Cholesterol risk score
        X['cholesterol_risk'] = X['ldl_cholesterol'] / X['hdl_cholesterol']

        # BMI Physical Activity Interaction
        X['bmi_activity_interaction'] = X['bmi'] * X['physical_activity_minutes_per_week']

        # Screen time x Sleep
        X['screen_sleep_interaction'] = X['screen_time_hours_per_day'] * X['sleep_hours_per_day']


        # Domain driven derived features

        # bmi categories
        X['bmi_category'] = pd.cut(X['bmi'], bins=[0,18.5,25,30,100], labels=['under','normal','over','obese'])

        # Age group
        X['age_group'] = pd.cut(X['age'], bins=[0, 30, 45, 60, 100], labels=['young', 'mid_age', 'senior', 'elderly'])

        # Sleep Quality Category
        X['sleep_quality'] = pd.cut(X['sleep_hours_per_day'], bins=[0, 5, 7, 9, 24], labels=['poor', 'fair', 'good', 'excellent'])

        #Physical Activity Flag
        X['activity_level'] = pd.cut(X['physical_activity_minutes_per_week'], bins=[0, 150, 300, 10000], labels=['low', 'moderate', 'high'])
        # X = pd.get_dummies(X, columns=['activity_level'], drop_first=True)

        #Waist to hip Ratio Threshold
        X['abdominal_obesity'] = (X['waist_to_hip_ratio'] > 0.90).astype(int)

        #Hypertension Flag
        X['hypertension_flag'] = ((X['systolic_bp'] >= 130) | (X['diastolic_bp'] >= 80)).astype(int)

        #High Cholesterol Risk Flag
        X['high_cholesterol_flag'] = (X['cholesterol_total'] >= 240).astype(int)

        return X

##### Encoding the binary features

In [ ]:
# binary_features = [
#     'abdominal_obesity','hypertension_flag','high_cholesterol_flag',
#     'family_history_diabetes','hypertension_history','cardiovascular_history'
# ]

# for col in binary_features:
#     X[col] = X[col].astype(int)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV


#### Numeric Pipeline

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#### Categorical Pipeline

In [ ]:
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

#### Combining Preprocessing 

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features)
    ]
)

#### Baseline Model - LogisticRegression

In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ('feature_engineering', FeatureEngineering()),
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(
        max_iter=100,
        class_weight='balanced',
        random_state=42
    ))
])

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=1
    ))
])

rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_grid,
    n_iter=15,                 # start small
    scoring='roc_auc',
    cv=3,                      # reduce CV folds
    verbose=2,
    random_state=42,
    n_jobs=-1                  # parallelize CV safely
)

In [ ]:
#Training 
log_reg_pipeline.fit(X_train, y_train)

In [ ]:
rf_random_search.fit(X_train, y_train)

In [ ]:
#Evaluation on validation data

from sklearn.metrics import roc_auc_score

y_pred_proba_lr = log_reg_pipeline.predict_proba(X_valid)[:, 1]
# y_pred_proba_rf = rf_model.predict_proba(X_valid)[:, 1]


roc_auc_lr = roc_auc_score(y_valid, y_pred_proba_lr)
# roc_auc_rf = roc_auc_score(y_valid, y_pred_proba_rf)

print("Logistic Regression ROC AUC:", roc_auc_lr)
# print("Random Forest ROC AUC: ", roc_auc_rf)

In [ ]:
best_rf = rf_random_search.best_estimator_

In [ ]:
print("Best ROC AUC:", rf_random_search.best_score_)
print("Best Parameters:")
print(rf_random_search.best_params_)

In [ ]:
test_data = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv", index_col = 0)

In [ ]:
# test_proba = rf_model.predict_proba(test_data)[:, 1]
test_proba = best_rf.predict_proba(test_data)[:, 1]

In [ ]:
submission = pd.DataFrame({
    "id": test_data.index,   # or use an existing ID column if present
    "diagnosed_diabetes": test_proba
})

In [ ]:
submission.to_csv("submission.csv", index=False)